# 02_run_grouthtruth_simulated

This notebook gets the grouth truthed simulated assignments.

## Set-up

In [0]:
%python
import pandas as pd
from pathlib import Path

In [0]:
%sh

# BLAST+ installed
apt-get -q install -y ncbi-blast+

In [0]:
# Install packages
installed <- rownames(installed.packages())

ensure_cran <- function(pkgs, repos = "https://www.stats.bris.ac.uk/R/") {
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    install.packages(to_install, dependencies = TRUE, repos = repos)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

ensure_bioc <- function(pkgs) {
  if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
  }
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    BiocManager::install(to_install, ask = FALSE, update = FALSE)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

## CRAN packages
cran_pkgs <- c("dplyr", "taxize", "rentrez", "stringr", "taxonomizr")
ensure_cran(cran_pkgs)

## Bioconductor packages
bioc_pkgs <- c("Biostrings")
ensure_bioc(bioc_pkgs)

## Create grouth truth databases

We need to create ground truthed reference databases for simulated data in a BLAST format. This has to be done on a per sample (or mock community) basis so that the exact sequences get matched.

We first need to tidy the fasta files provided by EBI since the labels are a bit messy.

In [0]:
# Function to parse headers

parse_header <- function(header){

  # remove leading sequence number if present
  header_clean <- sub("^\\d+\\s+", "", header)

  words <- strsplit(header_clean, "\\s+")[[1]]

  accession <- words[1]

  species <- NA

  if(length(words) >= 3){

    for(i in 2:(length(words)-1)){

      w1 <- words[i]
      w2 <- words[i + 1]

      if(
        grepl("^[A-Z][a-z]+$", w1) &&
        grepl("^[a-z-]+$", w2)
      ){

        species <- paste(w1, w2)
        break

      }
    }
  }

  data.frame(
    accession = accession,
    species = species,
    stringsAsFactors = FALSE
  )
}

# ========================================
# Process all simulated databases
# ========================================

db_dir <- "Data/Databases/Simulated_GroundTruth"

for(i in 1:10){

  fasta_file <- file.path(
    db_dir,
    paste0("riaz_EA_", i, ".fasta")
  )

  cat("\nProcessing:", fasta_file, "\n")

  fasta <- readDNAStringSet(fasta_file)

  headers <- names(fasta)

  taxonomy_lookup <- bind_rows(
    lapply(headers, parse_header)
  ) %>%
    mutate(
      genus = word(species, 1)
    )

  # records requiring review
  problem_records <- taxonomy_lookup %>%
    filter(is.na(species))

  cat(
    "Parsed",
    nrow(taxonomy_lookup),
    "records\n"
  )

  cat(
    nrow(problem_records),
    "records require manual review\n"
  )

  # save lookup table
  write.csv(
    taxonomy_lookup,
    file.path(
      db_dir,
      paste0("riaz_EA_", i, "_taxonomy_lookup.csv")
    ),
    row.names = FALSE
  )

  # save problematic records separately
  if(nrow(problem_records) > 0){

    write.csv(
      problem_records,
      file.path(
        db_dir,
        paste0("riaz_EA_", i, "_problem_records.csv")
      ),
      row.names = FALSE
    )

  }

}

In [0]:
db_dir <- "Data/Databases/Simulated_GroundTruth"

for(i in 1:10){

  fasta_file <- file.path(
    db_dir,
    paste0("riaz_EA_", i, ".fasta")
  )

  lookup_file <- file.path(
    db_dir,
    paste0("riaz_EA_", i, "_taxonomy_lookup.csv")
  )

  output_fasta <- file.path(
    db_dir,
    paste0("riaz_EA_", i, "_taxonomy.fasta")
  )

  cat("Processing:", fasta_file, "\n")

  fasta <- readDNAStringSet(fasta_file)

  lookup <- read.csv(
    lookup_file,
    stringsAsFactors = FALSE
  )

  accession_to_species <- setNames(
    lookup$species,
    lookup$accession
  )

  new_headers <- sapply(names(fasta), function(header){

    header_clean <- sub("^\\d+\\s+", "", header)

    accession <- strsplit(header_clean, "\\s+")[[1]][1]

    species <- accession_to_species[[accession]]

    if(is.null(species) || is.na(species)){

      accession

    } else {

      paste0(
        accession,
        "|",
        gsub(" ", "_", species)
      )

    }

  })

  names(fasta) <- new_headers

  writeXStringSet(
    fasta,
    output_fasta
  )

}

In [0]:
%sh
BASE_DIR="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/Simulated_GroundTruth"

for i in {1..10}
do
    FASTA="${BASE_DIR}/riaz_EA_${i}_taxonomy.fasta"
    OUTDIR="${BASE_DIR}/riaz_EA_${i}_db"

    mkdir -p "${OUTDIR}"

    makeblastdb \
        -in "${FASTA}" \
        -dbtype nucl \
        -out "${OUTDIR}/riaz_EA_${i}_db"
done

## Create BLAST input queries

BLAST queries need to be seperated by sample for ground truthing.

In [0]:
%python

base = Path("/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data")
processed_dir = base / "Processed" / "Simulated"

counts_file = processed_dir / "06_ASV_counts_DADA2.tsv"
out_dir = processed_dir / "blast_queries_DADA2"

out_dir.mkdir(exist_ok=True)

df = pd.read_csv(counts_file, sep="\t", index_col=0)

for sample in df.columns:

    sample_seqs = df.index[df[sample] > 0]

    fasta_file = out_dir / f"{sample}.fasta"

    with open(fasta_file, "w") as f:
        for i, seq in enumerate(sample_seqs, start=1):
            f.write(f">ASV_{i}\n")
            f.write(f"{seq}\n")

In [0]:
%python

counts_file = processed_dir / "06b_ASV_counts_VSEARCH.tsv"
out_dir = processed_dir / "blast_queries_VSEARCH"

out_dir.mkdir(exist_ok=True)

df = pd.read_csv(counts_file, sep="\t", index_col=0)

for sample in df.columns:

    sample_seqs = df.index[df[sample] > 0]

    fasta_file = out_dir / f"{sample}.fasta"

    with open(fasta_file, "w") as f:
        for i, seq in enumerate(sample_seqs, start=1):
            f.write(f">ASV_{i}\n")
            f.write(f"{seq}\n")

## Run BLAST

Now run BLAST for each sample and each sequence inference.

In [0]:
%sh
bash Scripts/08g_assign_taxonomy_BLAST_DADA2_simulatedGT.sh

In [0]:
blast_dir <- "Data/Processed/Simulated/08_groundtruth_BLAST_DADA2"

blast_files <- list.files(
  blast_dir,
  pattern = "\\.txt$",
  full.names = TRUE
)

cat("Found", length(blast_files), "files\n")

for(blast_file in blast_files){

  cat("Processing:", blast_file, "\n")

  blast_out <- read.table(
    blast_file,
    sep = "\t",
    header = FALSE,
    stringsAsFactors = FALSE
  )

  cat("Rows:", nrow(blast_out), "\n")

  split_tax <- str_split_fixed(blast_out$V2, "\\|", 2)

  blast_out$V2 <- split_tax[,1]
  blast_out$V3 <- paste(
    split_tax[,1],
    gsub("_", " ", split_tax[,2])
  )

  write.table(
    blast_out,
    blast_file,
    sep = "\t",
    quote = FALSE,
    row.names = FALSE,
    col.names = FALSE
  )
}

In [0]:
%sh
bash Scripts/08g_assign_taxonomy_BLAST_VSEARCH_simulatedGT.sh

In [0]:
blast_dir <- "Data/Processed/Simulated/08_groundtruth_BLAST_VSEARCH"

blast_files <- list.files(
  blast_dir,
  pattern = "\\.txt$",
  full.names = TRUE
)

cat("Found", length(blast_files), "files\n")

for(blast_file in blast_files){

  cat("Processing:", blast_file, "\n")

  blast_out <- read.table(
    blast_file,
    sep = "\t",
    header = FALSE,
    stringsAsFactors = FALSE
  )

  cat("Rows:", nrow(blast_out), "\n")

  split_tax <- str_split_fixed(blast_out$V2, "\\|", 2)

  blast_out$V2 <- split_tax[,1]
  blast_out$V3 <- paste(
    split_tax[,1],
    gsub("_", " ", split_tax[,2])
  )

  write.table(
    blast_out,
    blast_file,
    sep = "\t",
    quote = FALSE,
    row.names = FALSE,
    col.names = FALSE
  )
}

## Condense BLAST outputs

Now let's condense the ground truth simulated BLAST outputs.

In [0]:
min_pident <- 95
min_length <- 88 # 20% within 110 amplicon length
top_frac <- 0.95

In [0]:
path <- getwd()

blast_dir <- file.path(
  path,
  "Data",
  "Processed",
  "Simulated",
  "08_groundtruth_BLAST_DADA2"
)

db_dir <- file.path(
  path,
  "Data",
  "Databases",
  "Simulated_GroundTruth"
)

out_dir <- file.path(
  path,
  "Data",
  "Processed",
  "Simulated",
  "Simulated_GT"
)

all_lca_results <- list()

# =====================================================
# FIND BLAST FILES
# =====================================================

blast_files <- list.files(
  blast_dir,
  pattern = "^08_riaz_EA_.*_blast\\.txt$",
  full.names = TRUE
)

cat("Found", length(blast_files), "blast files\n")

# =====================================================
# LOOP THROUGH FILES
# =====================================================

for(blast_file in blast_files){

  sample <- sub(
    "^08_(riaz_EA_[0-9]+)_blast\\.txt$",
    "\\1",
    basename(blast_file)
  )

  cat("\nProcessing:", sample, "\n")

  # ===================================================
  # Lookup table
  # ===================================================

  lookup_file <- file.path(
    db_dir,
    paste0(sample, "_taxonomy_lookup.csv")
  )

  if(!file.exists(lookup_file)){
    warning("Missing lookup file: ", lookup_file)
    next
  }

  taxonomy_lookup <- read.csv(
    lookup_file,
    stringsAsFactors = FALSE
  )

  # ===================================================
  # BLAST output
  # ===================================================

  blast_out <- read.table(
    blast_file,
    sep = "\t",
    header = FALSE,
    stringsAsFactors = FALSE
  )

  colnames(blast_out) <- c(
    "qseqid",
    "sseqid",
    "species",
    "pident",
    "length",
    "mismatch",
    "gapopen",
    "qstart",
    "qend",
    "sstart",
    "send",
    "evalue",
    "bitscore"
  )

  # Remove accession from title

  blast_out <- blast_out %>%
    mutate(
      species = sub("^[^ ]+ ", "", species)
    )

  # ===================================================
  # Join taxonomy
  # ===================================================

  blast_out <- blast_out %>%
    left_join(
      taxonomy_lookup,
      by = c("sseqid" = "accession")
    )

  # ===================================================
  # Filter
  # ===================================================

  blast_out <- blast_out %>%
    filter(pident >= min_pident) %>%
    filter(length >= min_length) %>%
    group_by(qseqid) %>%
    mutate(
      max_bitscore = max(bitscore)
    ) %>%
    filter(
      bitscore >= (top_frac * max_bitscore)
    ) %>%
    ungroup()

  if(nrow(blast_out) == 0){
    warning("No hits remain for ", sample)
    next
  }

  # ===================================================
  # Taxonomy matrix
  # ===================================================

  taxa <- blast_out %>%
    select(
      genus,
      species.y
    )

  colnames(taxa) <- c(
    "Genus",
    "Species"
  )

  combined_data <- cbind(
    ASV_ID = blast_out$qseqid,
    as.data.frame(taxa)
  )

  # ===================================================
  # LCA
  # ===================================================

  lca_results <- combined_data %>%
    group_by(ASV_ID) %>%
    do({

      sub_matrix <- .[, -1] %>%
        distinct() %>%
        as.matrix()

      lca <- taxonomizr::condenseTaxa(sub_matrix)

      as.data.frame(lca)

    })

  # ===================================================
  # Format
  # ===================================================

  lca_results[lca_results == "NA"] <- NA

  lca_results <- lca_results %>%
    ungroup() %>%
    mutate(
      Sample = sample,

      lca_rank = apply(
        .[, -1],
        1,
        function(x){

          idx <- which(
            !is.na(x) &
              x != ""
          )

          if(length(idx) > 0){

            colnames(lca_results)[-1][max(idx)]

          } else {

            NA

          }

        }
      ),

      lca_name = apply(
        .[, -1],
        1,
        function(x){

          idx <- which(
            !is.na(x) &
              x != ""
          )

          if(length(idx) > 0){

            x[max(idx)]

          } else {

            NA

          }

        }
      )
    ) %>%
    select(
      Sample,
      ASV_ID,
      everything(),
      lca_rank,
      lca_name
    )

  # ===================================================
  # Save sample result
  # ===================================================

  out_file <- file.path(
    blast_dir,
    paste0(
      "09_",
      sample,
      "_LCA_results.tsv"
    )
  )

  write.table(
    lca_results,
    out_file,
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )

  all_lca_results[[sample]] <- lca_results

}

# =====================================================
# COMBINED OUTPUT
# =====================================================

combined_lca <- bind_rows(all_lca_results)

write.table(
  combined_lca,
  file.path(
    out_dir,
    "09_DADA2_ASV_LCA_results_EA_riaz95_88_top_0.95%.tsv"
  ),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

cat(
  "\nFinished processing",
  length(all_lca_results),
  "samples\n"
)

In [0]:
path <- getwd()

blast_dir <- file.path(
  path,
  "Data",
  "Processed",
  "Simulated",
  "08_groundtruth_BLAST_VSEARCH"
)

all_lca_results <- list()

# =====================================================
# FIND BLAST FILES
# =====================================================

blast_files <- list.files(
  blast_dir,
  pattern = "^08_riaz_EA_.*_blast\\.txt$",
  full.names = TRUE
)

cat("Found", length(blast_files), "blast files\n")

# =====================================================
# LOOP THROUGH FILES
# =====================================================

for(blast_file in blast_files){

  sample <- sub(
    "^08_(riaz_EA_[0-9]+)_blast\\.txt$",
    "\\1",
    basename(blast_file)
  )

  cat("\nProcessing:", sample, "\n")

  # ===================================================
  # Lookup table
  # ===================================================

  lookup_file <- file.path(
    db_dir,
    paste0(sample, "_taxonomy_lookup.csv")
  )

  if(!file.exists(lookup_file)){
    warning("Missing lookup file: ", lookup_file)
    next
  }

  taxonomy_lookup <- read.csv(
    lookup_file,
    stringsAsFactors = FALSE
  )

  # ===================================================
  # BLAST output
  # ===================================================

  blast_out <- read.table(
    blast_file,
    sep = "\t",
    header = FALSE,
    stringsAsFactors = FALSE
  )

  colnames(blast_out) <- c(
    "qseqid",
    "sseqid",
    "species",
    "pident",
    "length",
    "mismatch",
    "gapopen",
    "qstart",
    "qend",
    "sstart",
    "send",
    "evalue",
    "bitscore"
  )

  # Remove accession from title

  blast_out <- blast_out %>%
    mutate(
      species = sub("^[^ ]+ ", "", species)
    )

  # ===================================================
  # Join taxonomy
  # ===================================================

  blast_out <- blast_out %>%
    left_join(
      taxonomy_lookup,
      by = c("sseqid" = "accession")
    )

  # ===================================================
  # Filter
  # ===================================================

  blast_out <- blast_out %>%
    filter(pident >= min_pident) %>%
    filter(length >= min_length) %>%
    group_by(qseqid) %>%
    mutate(
      max_bitscore = max(bitscore)
    ) %>%
    filter(
      bitscore >= (top_frac * max_bitscore)
    ) %>%
    ungroup()

  if(nrow(blast_out) == 0){
    warning("No hits remain for ", sample)
    next
  }

  # ===================================================
  # Taxonomy matrix
  # ===================================================

  taxa <- blast_out %>%
    select(
      genus,
      species.y
    )

  colnames(taxa) <- c(
    "Genus",
    "Species"
  )

  combined_data <- cbind(
    ASV_ID = blast_out$qseqid,
    as.data.frame(taxa)
  )

  # ===================================================
  # LCA
  # ===================================================

  lca_results <- combined_data %>%
    group_by(ASV_ID) %>%
    do({

      sub_matrix <- .[, -1] %>%
        distinct() %>%
        as.matrix()

      lca <- taxonomizr::condenseTaxa(sub_matrix)

      as.data.frame(lca)

    })

  # ===================================================
  # Format
  # ===================================================

  lca_results[lca_results == "NA"] <- NA

  lca_results <- lca_results %>%
    ungroup() %>%
    mutate(
      Sample = sample,

      lca_rank = apply(
        .[, -1],
        1,
        function(x){

          idx <- which(
            !is.na(x) &
              x != ""
          )

          if(length(idx) > 0){

            colnames(lca_results)[-1][max(idx)]

          } else {

            NA

          }

        }
      ),

      lca_name = apply(
        .[, -1],
        1,
        function(x){

          idx <- which(
            !is.na(x) &
              x != ""
          )

          if(length(idx) > 0){

            x[max(idx)]

          } else {

            NA

          }

        }
      )
    ) %>%
    select(
      Sample,
      ASV_ID,
      everything(),
      lca_rank,
      lca_name
    )

  # ===================================================
  # Save sample result
  # ===================================================

  out_file <- file.path(
    blast_dir,
    paste0(
      "09_",
      sample,
      "_LCA_results.tsv"
    )
  )

  write.table(
    lca_results,
    out_file,
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
  )

  all_lca_results[[sample]] <- lca_results

}

# =====================================================
# COMBINED OUTPUT
# =====================================================

combined_lca <- bind_rows(all_lca_results)

write.table(
  combined_lca,
  file.path(
    out_dir,
    "08_vsearch_output_vsearch_EA_riaz_lca.tsv"
  ),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

cat(
  "\nFinished processing",
  length(all_lca_results),
  "samples\n"
)